# Atelier 3.A — Un agent ReAct sans framework

**Durée : 45 min**

Avant d'utiliser un framework, on écrit la boucle d'un agent à la main, en quelques dizaines de lignes :

1. le modèle reçoit la question et la description des outils ;
2. il répond, ou il demande d'appeler un outil (*function calling*) ;
3. le programme exécute l'outil et renvoie le résultat au modèle ;
4. on recommence jusqu'à une réponse finale, ou jusqu'à la limite d'itérations.

Trois outils : une calculatrice, les taux de change de la BCE (API Frankfurter) et la recherche sur Wikipédia. Les deux API sont publiques et gratuites.

In [ ]:
import ast
import json
import operator
import re

import httpx
import outils_llm as llm

MODELE = llm.choisir_modele()   # le modèle doit savoir appeler des outils (gpt-oss, Qwen 3.8, Nemotron 3.5…)
fournisseur, NOM = llm.decouper(MODELE)
client = llm.client(fournisseur)
print("Modèle :", MODELE)

## 1. Les outils : des fonctions Python ordinaires

Un outil renvoie un résultat **ou un message d'erreur exploitable** : le modèle doit comprendre ce qui n'a pas marché pour se corriger. La calculatrice n'utilise jamais `eval`, qui exécuterait n'importe quel code : elle n'accepte que des nombres et des opérateurs.

In [ ]:
OPERATEURS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
              ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculatrice(expression: str) -> dict:
    def evaluer(noeud):
        if isinstance(noeud, ast.Constant) and isinstance(noeud.value, (int, float)):
            return noeud.value
        if isinstance(noeud, ast.BinOp) and type(noeud.op) in OPERATEURS:
            return OPERATEURS[type(noeud.op)](evaluer(noeud.left), evaluer(noeud.right))
        if isinstance(noeud, ast.UnaryOp) and type(noeud.op) in OPERATEURS:
            return OPERATEURS[type(noeud.op)](evaluer(noeud.operand))
        raise ValueError("élément non autorisé")
    try:
        return {"resultat": round(evaluer(ast.parse(expression, mode="eval").body), 6)}
    except Exception as erreur:
        return {"erreur": f"Expression invalide ({erreur}). Utilisez des nombres et + - * / ** ( )."}

def taux_de_change(devise_source: str, devise_cible: str) -> dict:
    r = httpx.get("https://api.frankfurter.dev/v1/latest",
                  params={"base": devise_source.upper(), "symbols": devise_cible.upper()}, timeout=10)
    if r.status_code != 200:
        return {"erreur": f"Devise inconnue ou service indisponible (HTTP {r.status_code}). Codes ISO attendus : CHF, EUR, USD…"}
    donnees = r.json()
    taux = donnees["rates"][devise_cible.upper()]
    return {"taux": taux, "regle": f"1 {devise_source.upper()} = {taux} {devise_cible.upper()} : "
            f"montant en {devise_cible.upper()} = montant en {devise_source.upper()} × {taux}",
            "date": donnees["date"], "source": "BCE, via Frankfurter"}

# Wikimédia exige un User-Agent identifiant l'application et un contact (sinon : erreur 403).
USER_AGENT = "FormationAgentsIA/1.0 (https://www.example.org/formation; formation@example.org)"

def recherche_wikipedia(requete: str) -> dict:
    r = httpx.get("https://fr.wikipedia.org/w/api.php",
                  params={"action": "query", "list": "search", "srsearch": requete, "format": "json", "srlimit": 3},
                  headers={"User-Agent": USER_AGENT}, timeout=10)
    if r.status_code != 200 or "json" not in r.headers.get("content-type", ""):
        return {"erreur": f"Wikipédia indisponible (HTTP {r.status_code}). Réponds sans cette source et signale-le."}
    resultats = [{"titre": x["title"], "extrait": re.sub(r"<[^>]+>", "", x["snippet"])}
                 for x in r.json()["query"]["search"]]
    return {"resultats": resultats} if resultats else {"erreur": "Aucun résultat : reformulez la recherche."}

print(calculatrice("3 * 400 * 1.081"), taux_de_change("CHF", "EUR"), sep="\n")

## 2. Décrire les outils au modèle

Le modèle ne voit pas le code : il ne voit que le **nom**, la **description** et le **schéma des paramètres** (JSON Schema). C'est sur cette base qu'il choisit l'outil et ses arguments : la description fait partie du prompt.

In [ ]:
def schema(nom, description, parametres, obligatoires):
    return {"type": "function", "function": {"name": nom, "description": description,
            "parameters": {"type": "object", "properties": parametres, "required": obligatoires}}}

SCHEMAS = [
    schema("calculatrice", "Calcule une expression arithmétique (nombres, + - * / ** et parenthèses). À utiliser pour tout calcul.",
           {"expression": {"type": "string", "description": "Par exemple : 3 * 400 * 1.081"}}, ["expression"]),
    schema("taux_de_change", "Taux de change du jour publié par la Banque centrale européenne.",
           {"devise_source": {"type": "string", "description": "Code ISO, par exemple CHF"},
            "devise_cible": {"type": "string", "description": "Code ISO, par exemple EUR"}}, ["devise_source", "devise_cible"]),
    schema("recherche_wikipedia", "Recherche des articles de l'encyclopédie Wikipédia en français ; renvoie titres et extraits.",
           {"requete": {"type": "string", "description": "Mots-clés de la recherche"}}, ["requete"]),
]
OUTILS = {"calculatrice": calculatrice, "taux_de_change": taux_de_change, "recherche_wikipedia": recherche_wikipedia}

## 3. La boucle de l'agent

Toute la « mémoire » de l'agent est la liste `messages` : la question, chaque demande d'outil du modèle, chaque résultat. À chaque tour, on renvoie la liste entière.

In [ ]:
SYSTEME = "Tu es un assistant précis. Utilise les outils pour tout fait vérifiable et pour tout calcul. Réponds en français."

def agent(question, max_iterations=6, verbeux=True):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    tokens = 0
    for iteration in range(1, max_iterations + 1):
        reponse = client.chat.completions.create(model=NOM, messages=messages, tools=SCHEMAS)
        tokens += reponse.usage.total_tokens
        message = reponse.choices[0].message
        if not message.tool_calls:                                      # réponse finale
            if verbeux:
                print(f"✅ Réponse après {iteration} appel(s) au modèle, {tokens} tokens :\n{message.content}")
            return message.content, messages
        messages.append({"role": "assistant", "content": message.content or "",
                         "tool_calls": [{"id": a.id, "type": "function",
                                         "function": {"name": a.function.name, "arguments": a.function.arguments}}
                                        for a in message.tool_calls]})
        for appel in message.tool_calls:                                # exécuter chaque outil demandé
            arguments = json.loads(appel.function.arguments or "{}")
            fonction = OUTILS.get(appel.function.name)
            resultat = fonction(**arguments) if fonction else {"erreur": f"Outil inconnu : {appel.function.name}"}
            if verbeux:
                print(f"🤖 {appel.function.name}({arguments})\n🔧 {resultat}\n")
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": json.dumps(resultat, ensure_ascii=False)})
    if verbeux:
        print(f"⛔ Limite de {max_iterations} itérations atteinte sans réponse finale.")
    return None, messages

## 4. À l'essai

In [ ]:
reponse, messages = agent("Un client allemand demande le prix TTC, en euros, de 3 imprimantes LX-500 à 400 CHF HT "
                          "l'unité (TVA suisse de 8,1 %). Combien cela fait-il ?")

In [ ]:
reponse, messages = agent("En quelle année la nouvelle loi suisse sur la protection des données (nLPD) est-elle "
                          "entrée en vigueur, et combien d'années se sont écoulées jusqu'en 2026 ?")

**À vérifier.** 1 CHF vaut un peu plus de 1 EUR : le prix en euros doit donc être **plus élevé** que le prix en francs. Le modèle a-t-il multiplié par le taux ? Une première version de l'outil renvoyait le taux seul, et le modèle divisait parfois au lieu de multiplier. L'outil renvoie désormais la règle de calcul : la conception de l'outil prévient l'erreur du modèle.

**À observer**
- L'ordre des appels n'est écrit nulle part : c'est le modèle qui **planifie** (taux de change, puis calcul, ou l'inverse).
- Les résultats des outils reviennent dans le contexte : le modèle **observe** avant d'agir à nouveau. C'est le schéma ReAct : raisonner, agir, observer.

La liste `messages` est la trace complète de l'exécution :

In [ ]:
for m in messages:
    if m["role"] == "assistant":
        print("assistant →", [a["function"]["name"] for a in m["tool_calls"]])
    else:
        print(f"{m['role']:9} :", str(m["content"])[:110].replace("\n", " "))

## 5. Un garde-fou : la limite d'itérations

Sans limite, un agent qui tourne en rond consomme des tokens indéfiniment. Avec une limite d'un seul appel, l'agent ne peut pas terminer :

In [ ]:
reponse, _ = agent("Combien font 1 250 CHF en dollars américains ?", max_iterations=1)

## 6. Ajouter un outil

Ajouter une capacité, c'est écrire une fonction et sa description. Ici, la météo actuelle d'une ville, avec l'API publique Open-Meteo (géocodage, puis prévisions).

In [ ]:
def meteo(ville: str) -> dict:
    lieux = httpx.get("https://geocoding-api.open-meteo.com/v1/search",
                      params={"name": ville, "count": 1, "language": "fr"}, timeout=10).json().get("results")
    if not lieux:
        return {"erreur": f"Ville inconnue : {ville}."}
    lieu = lieux[0]
    actuel = httpx.get("https://api.open-meteo.com/v1/forecast",
                       params={"latitude": lieu["latitude"], "longitude": lieu["longitude"],
                               "current": "temperature_2m,wind_speed_10m"}, timeout=10).json()["current"]
    return {"ville": lieu["name"], "pays": lieu.get("country"), "temperature_c": actuel["temperature_2m"],
            "vent_kmh": actuel["wind_speed_10m"], "heure": actuel["time"]}

SCHEMAS.append(schema("meteo", "Météo actuelle d'une ville : température en °C et vent en km/h.",
                      {"ville": {"type": "string", "description": "Nom de la ville, par exemple Lausanne"}}, ["ville"]))
OUTILS["meteo"] = meteo

reponse, _ = agent("Quel temps fait-il à Lausanne en ce moment ? Donne aussi la température en degrés Fahrenheit.")

✏️ Posez une question qui exige trois outils à la suite. Puis dégradez volontairement une description (par exemple « Fait un calcul. » → « Outil. ») et observez le comportement de l'agent.

## À retenir

- Un agent, c'est une **boucle** : le modèle choisit, le programme exécute, le résultat revient au modèle.
- Le modèle ne voit que les **noms, descriptions et schémas** des outils : leur rédaction est décisive.
- Un outil renvoie des erreurs explicites ; il n'exécute jamais de code arbitraire (pas d'`eval`).
- Une **limite d'itérations** est indispensable. Les frameworks (atelier 3.B) ajoutent l'état, la mémoire, la reprise et la validation humaine.